# AQI Hybrid Model: SARIMA Error-Correction Features for XGBoost

This notebook updates the SARIMA -> XGBoost pipeline by engineering SARIMA residual features so XGBoost can learn targeted error correction.

## 1) Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2) Load Data and Select City

In [2]:
df_raw = pd.read_csv('Dataset.csv')
df_raw = df_raw.rename(columns={'City': 'city', 'Date': 'date'})
df_raw['date'] = pd.to_datetime(df_raw['date'])

aqi_counts = df_raw.groupby('city')['AQI'].apply(lambda s: s.notna().sum()).sort_values(ascending=False)
CITY = 'Ahmedabad'
if CITY not in aqi_counts.index or aqi_counts.get(CITY, 0) == 0:
    CITY = aqi_counts.index[0]

df = df_raw[df_raw['city'] == CITY].copy().sort_values('date').set_index('date')

TARGET = 'AQI'
POLLUTANTS = [c for c in ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3'] if c in df.columns]
WEATHER = [c for c in ['t2m', 'd2m', 'sp', 'blh', 'u10', 'v10'] if c in df.columns]

keep_cols = POLLUTANTS + WEATHER + [TARGET]
df = df[keep_cols].copy()

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

feature_fill_cols = [c for c in df.columns if c != TARGET]
df[feature_fill_cols] = df[feature_fill_cols].ffill().bfill().fillna(0)
df = df.dropna(subset=[TARGET]).copy()

df = df.asfreq('D')
df[feature_fill_cols] = df[feature_fill_cols].ffill().bfill().fillna(0)
df = df.dropna(subset=[TARGET]).copy()

print('City:', CITY)
print('Rows:', len(df))
print('Date range:', df.index.min(), 'to', df.index.max())

City: Ahmedabad
Rows: 1334
Date range: 2015-01-29 00:00:00 to 2020-07-01 00:00:00


## 3) Base Feature Engineering (Pollutants, Weather, AQI Lags, Rolling, Time)

In [3]:
for lag in [1, 2, 3, 7, 14, 21, 28]:
    df[f'AQI_lag_{lag}'] = df[TARGET].shift(lag)

for window in [7, 14, 30]:
    shifted = df[TARGET].shift(1)
    df[f'AQI_roll_{window}_mean'] = shifted.rolling(window).mean()
    df[f'AQI_roll_{window}_std'] = shifted.rolling(window).std()

df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['day_of_year'] = df.index.dayofyear
df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df = df.dropna().copy()
print('Rows after base features:', len(df))

Rows after base features: 1304


## 4) Temporal Train/Test Split (80/20)

In [4]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print('Train rows:', len(train_df), '| Test rows:', len(test_df))
print('Train range:', train_df.index.min(), 'to', train_df.index.max())
print('Test range :', test_df.index.min(), 'to', test_df.index.max())

Train rows: 1043 | Test rows: 261
Train range: 2015-03-04 00:00:00 to 2019-10-06 00:00:00
Test range : 2019-10-07 00:00:00 to 2020-07-01 00:00:00


## 5) Fit SARIMA and Build Error-Correction Features

Requested steps implemented:
- `sarima_error = actual - sarima_pred`
- `sarima_error_lag_1`, `sarima_error_lag_2`, `sarima_error_roll_mean_3`
- strict shift to prevent leakage
- seamless train->test transition using one combined error timeline

In [5]:
SARIMA_ORDER = (1, 1, 1)
SEASONAL_ORDER = (1, 1, 1, 7)

sarima_model = SARIMAX(
    train_df[TARGET],
    order=SARIMA_ORDER,
    seasonal_order=SEASONAL_ORDER,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False, maxiter=200)

# In-sample train prediction and out-of-sample test forecast
sarima_pred_train = sarima_fit.fittedvalues.reindex(train_df.index)
sarima_pred_test = sarima_fit.forecast(steps=len(test_df))
sarima_pred_test.index = test_df.index

# Combined SARIMA prediction timeline for seamless error-feature engineering
sarima_pred_full = pd.concat([sarima_pred_train, sarima_pred_test]).sort_index()
sarima_pred_full = sarima_pred_full.reindex(df.index)

# 1) Residuals
df['sarima_pred'] = sarima_pred_full
df['sarima_error'] = df[TARGET] - df['sarima_pred']

# 2) Error features (strictly leakage-safe with shift)
df['sarima_error_lag_1'] = df['sarima_error'].shift(1)
df['sarima_error_lag_2'] = df['sarima_error'].shift(2)
df['sarima_error_roll_mean_3'] = df['sarima_error'].shift(1).rolling(window=3).mean()

# 3) Handle NaNs introduced by lagging/rolling
df = df.dropna(subset=['sarima_error_lag_1', 'sarima_error_lag_2', 'sarima_error_roll_mean_3']).copy()

print('Rows after SARIMA error features:', len(df))
display(df[['sarima_pred', 'sarima_error', 'sarima_error_lag_1', 'sarima_error_lag_2', 'sarima_error_roll_mean_3']].head())

c:\Users\piyus\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\piyus\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Rows after SARIMA error features: 1301


c:\Users\piyus\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


,sarima_pred,sarima_error,sarima_error_lag_1,sarima_error_lag_2,sarima_error_roll_mean_3
date,,,,,
2015-03-09,71.946602,161.053398,-703.729467,334.215200,4.495245
2015-03-10,214.753884,82.246116,161.053398,-703.729467,-69.486956
2015-03-11,308.357128,21.642872,82.246116,161.053398,-153.476651
2015-03-12,340.935163,-88.935163,21.642872,82.246116,88.314129
2015-03-13,499.552513,-255.552513,-88.935163,21.642872,4.984608


## 6) Feature Fusion for XGBoost

In [6]:
# Rebuild split after dropped rows to keep exact alignment
split_idx2 = int(len(df) * 0.8)
train_df2 = df.iloc[:split_idx2].copy()
test_df2 = df.iloc[split_idx2:].copy()

error_feature_cols = ['sarima_error_lag_1', 'sarima_error_lag_2', 'sarima_error_roll_mean_3']

exclude_cols = [TARGET, 'sarima_error']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X_train = train_df2[feature_cols].copy()
X_test = test_df2[feature_cols].copy()
y_train = train_df2[TARGET].copy()
y_test = test_df2[TARGET].copy()

print('Train rows:', len(X_train), '| Test rows:', len(X_test))
print('Total features:', len(feature_cols))
print('Error-correction features included:', all(c in feature_cols for c in error_feature_cols))

Train rows: 1040 | Test rows: 261
Total features: 34
Error-correction features included: True


## 7) Updated XGBoost Training + Evaluation (R² and MSE)

In [7]:
# Optional baseline (without error features) for comparison
base_feature_cols = [c for c in feature_cols if c not in error_feature_cols]

imputer = SimpleImputer(strategy='median')
X_train_base = pd.DataFrame(imputer.fit_transform(X_train[base_feature_cols]), columns=base_feature_cols, index=X_train.index)
X_test_base = pd.DataFrame(imputer.transform(X_test[base_feature_cols]), columns=base_feature_cols, index=X_test.index)

X_train_hybrid = pd.DataFrame(imputer.fit_transform(X_train[feature_cols]), columns=feature_cols, index=X_train.index)
X_test_hybrid = pd.DataFrame(imputer.transform(X_test[feature_cols]), columns=feature_cols, index=X_test.index)

xgb_params = dict(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=RANDOM_SEED,
    n_jobs=-1
)

base_model = xgb.XGBRegressor(**xgb_params)
base_model.fit(X_train_base, y_train, verbose=False)
base_pred = base_model.predict(X_test_base)

hybrid_model = xgb.XGBRegressor(**xgb_params)
hybrid_model.fit(X_train_hybrid, y_train, verbose=False)
hybrid_pred = hybrid_model.predict(X_test_hybrid)

base_mse = mean_squared_error(y_test, base_pred)
base_r2 = r2_score(y_test, base_pred)

hybrid_mse = mean_squared_error(y_test, hybrid_pred)
hybrid_r2 = r2_score(y_test, hybrid_pred)

print('XGBoost Baseline (no SARIMA-error features)')
print(f'  MSE : {base_mse:.4f}')
print(f'  R^2 : {base_r2:.4f}')

print('XGBoost Hybrid (with SARIMA-error features)')
print(f'  MSE : {hybrid_mse:.4f}')
print(f'  R^2 : {hybrid_r2:.4f}')

XGBoost Baseline (no SARIMA-error features)
  MSE : 14588.1287
  R^2 : 0.8017
XGBoost Hybrid (with SARIMA-error features)
  MSE : 11770.7259
  R^2 : 0.8400


## 8) Compact Comparison Table

In [8]:
comparison = pd.DataFrame({
    'Model': ['XGBoost Baseline', 'XGBoost + SARIMA Error Features'],
    'MSE': [base_mse, hybrid_mse],
    'R2': [base_r2, hybrid_r2]
})

print(comparison.round(4).to_string(index=False))

                          Model        MSE     R2
               XGBoost Baseline 14588.1287 0.8017
XGBoost + SARIMA Error Features 11770.7259 0.8400
